---
# **Cuaderno 2:**  Analisis de datos de ICGC
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: Agosto 2025

**Aims:**

* Analiza los datos obtenidos al filtrar por gen un archivo de mutaciones de la base de datos del ICGC

### Librerias

In [1]:
import pandas as pd

### Cargar archivo de mutaciones

El archivo proviene del cuaderno 01_Cancer_mutations

In [56]:
#Nombre grupo de genes
grupo = "tgfb"
#Ruta 
ruta = "../data/Cancer/mut_cancer_{0}.csv".format(grupo)
#Cargar archivo
df = pd.read_csv(ruta)
df = df.drop_duplicates()
#Datos
print(f"tamaño: \n{df.shape}")
print(f"valores nulos: \n{df.isnull().sum()}")


df.head(2)

tamaño: 
(33954, 8)
valores nulos: 
icgc_donor_id              0
uniprot_id                 0
gen_name                   0
aa_mutation            27339
tipo_cancer                0
transcript_affected        0
verification_status        0
consequence_type           0
dtype: int64


,icgc_donor_id,uniprot_id,gen_name,aa_mutation,tipo_cancer,transcript_affected,verification_status,consequence_type
0,DO37659,Q13873,BMPR2,NaN,Piel,ENST00000374574,not tested,3_prime_UTR_variant
1,DO37659,Q13873,BMPR2,P978P,Piel,ENST00000374580,not tested,synonymous_variant


## 1. Procesar archivo de mutaciones


In [57]:
df_cancer = df.dropna()
print(f"Tamaño: {df_cancer.shape}")
print(f"Número de pacientes: {df_cancer['icgc_donor_id'].nunique()}")
print(f"Número de genes: {df_cancer['gen_name'].nunique()}")
df_cancer

Tamaño: (6615, 8)
Número de pacientes: 1297
Número de genes: 12


,icgc_donor_id,uniprot_id,gen_name,aa_mutation,tipo_cancer,transcript_affected,verification_status,consequence_type
1,DO37659,Q13873,BMPR2,P978P,Piel,ENST00000374580,not tested,synonymous_variant
2,DO37571,Q13873,BMPR2,P516Q,Piel,ENST00000374580,not tested,missense_variant
3,DO37571,Q13873,BMPR2,P516Q,Piel,ENST00000374574,not tested,missense_variant
6,DO37551,Q13873,BMPR2,W484L,Piel,ENST00000374574,not tested,missense_variant
7,DO37551,Q13873,BMPR2,W484L,Piel,ENST00000374580,not tested,missense_variant
...,...,...,...,...,...,...,...,...
62768,DO555,P37173,TGFBR2,Q29E,Vejiga,ENST00000295754,not tested,missense_variant
62769,DO512,P37173,TGFBR2,M425I,Vejiga,ENST00000295754,not tested,missense_variant
62770,DO512,P37173,TGFBR2,M450I,Vejiga,ENST00000359013,not tested,missense_variant
62771,DO785,P37173,TGFBR2,A261R?,Vejiga,ENST00000295754,not tested,frameshift_variant


In [58]:
#Crea un data frame donde solo esten los datos que han sido verificados y que no esten testeados
df_verificado = df_cancer.loc[(df_cancer['verification_status'] == 'tested and verified') | (df_cancer['verification_status'] == 'not tested')]
print(f"Verificacion (df_cancer): {df_cancer['verification_status'].unique()}")
print(f"Verificacion (df_verificado): {df_verificado['verification_status'].unique()}")
print(f"Tamaño: {df_verificado.shape}")

Verificacion (df_cancer): ['not tested' 'tested and verified' 'tested and verified to be false']
Verificacion (df_verificado): ['not tested' 'tested and verified']
Tamaño: (6609, 8)


In [64]:
# elimina las mutaciones synonymous (silenciosas) 
df_consequence=df_verificado[df_verificado['consequence_type'] != 'synonymous_variant'].reset_index(drop=True)
print(f"Tamaño: {df_consequence.shape}")
print(f"Tipos de mutación: \n{df_consequence['consequence_type'].unique()}")


Tamaño: (5245, 8)
Tipos de mutación: 
['missense_variant' 'stop_gained' 'frameshift_variant' 'inframe_deletion'
 'stop_lost' 'stop_retained_variant' 'start_lost'
 'disruptive_inframe_insertion']


In [65]:
#Numero de mutaciones unicas y pacientes 
print(f"Número de mutaciones: {df_consequence['aa_mutation'].nunique()}")
print(f"Número de pacientes: {df_consequence['icgc_donor_id'].nunique()}")

Número de mutaciones: 2762
Número de pacientes: 1070


### Clasificación

In [69]:
# CLASIFICACIÓN DE MUTACIONES 
def clasificar_mutacion(gen, posicion_mutacion, cambio_residuo_simple, gene_domain_data, original_aa_mutation_str):
    """
    Clasifica una mutación dada su posición y el gen, basándose en la superposición con dominios/motivos de UniProt.
    'original_aa_mutation_str' es la cadena completa de la mutación (ej., "R200S" o "D403N403N") que se usará directamente.
    """
    resultado = {
        "gen": gen,
        "posicion": posicion_mutacion,
        "cambio_residuo": cambio_residuo_simple,
        "mutation_RefPosMut": original_aa_mutation_str, # Usamos la cadena original directamente
        "dominios_afectados_detallado": [],
    }

    gene_features = gene_domain_data.get(gen, {})
    return resultado

    overlapping_features = []
    for motif, list_of_ranges in gene_features.items():
        # Excluir motivos amplios y descriptivos como 'Cytoplasmic', 'Chain' y 'Alternative sequence' ---
        if any(keyword in motif for keyword in ["Cytoplasmic", "Chain", "Alternative sequence", "Modified residue_Phosphoserine", "Natural variant", "Turn"]):
            continue


        for rango_tuple in list_of_ranges:
            start_pos, end_pos = rango_tuple
            if start_pos <= posicion_mutacion <= end_pos:
                affected_feature_info = {
                    "motif_uniprot": motif,
                    "range": (start_pos, end_pos),
                    "categoria_clasificador": "Otro Dominio/Motivo" # Valor por defecto inicial
                }
                
                # --- LÓGICA DE MAPEO REFINADA ---
                if "Kinase" in motif or "Protein kinase" in motif:
                    affected_feature_info["categoria_clasificador"] = "Kinasa"
                elif "GS" in motif or "FKBP1A-binding" in motif:
                    affected_feature_info["categoria_clasificador"] = "GS"
                elif "Extracellular" in motif or "Propeptide" in motif or "Ectodominio" in motif:
                    affected_feature_info["categoria_clasificador"] = "Ectodominio"
                elif "Transmembrane" in motif:
                    affected_feature_info["categoria_clasificador"] = "Transmembrana"
               # elif "Binding site" in motif:
                #    affected_feature_info["categoria_clasificador"] = "Binding site"
               # elif "Active site" in motif:
                 #   affected_feature_info["categoria_clasificador"] = "Active site"
                elif "Signal" in motif:
                    affected_feature_info["categoria_clasificador"] = "Signal"
                
                overlapping_features.append(affected_feature_info)

    resultado["dominios_afectados_detallado"] = overlapping_features

    final_categorias_resumen = set()
    found_specific_category = False

    for affected_feature in overlapping_features:
        categoria = affected_feature["categoria_clasificador"]
        if categoria != "Otro Dominio/Motivo":
            final_categorias_resumen.add(categoria)
            found_specific_category = True
    
    resultado["dominios_clasificador_resumen"] = ", ".join(sorted(list(final_categorias_resumen)))
    
    if overlapping_features:
        resultado["rangos_uniprot_resumen"] = "; ".join(sorted([
            f"{f['motif_uniprot']}: [{f['range'][0]}-{f['range'][1]}]" for f in overlapping_features
        ]))
    else:
        resultado["rangos_uniprot_resumen"] = "N/A"


    return resultado

# CONFIGURACIÓN Y EJECUCIÓN PRINCIPAL

if __name__ == "__main__":
    print("--- INICIO DEL ANÁLISIS DE MUTACIONES ---")

    # --- A. CONFIGURACIÓN DE UNIPROT IDS ---
   
    uniprot_ids_for_genes = {
       "TGFBR1": "P36897", 
       "ACVR1": "Q04771",   
       "ACVR1B": "P36896",  
       "ACVR1C": "Q8NER5", 
       "ACVRL1": "P37023",  
       "BMPR1A": "P36894",  
       "BMPR1B": "O00238"   
    }

    gene_domain_data_from_uniprot = get_uniprot_features_as_dict_for_genes(uniprot_ids_for_genes)


    # CLASIFICACIÓN DE MUTACIONES DESDE DATAFRAME 
    print("\n--- Clasificación de Mutaciones desde DataFrame ---")

    RUTA_ARCHIVO_MUT = "../data/mutaciones/miss_mut"

    ARCHIVO = df_consequence

    
    try:
        #archivo_mutaciones = pd.read_csv(RUTA_ARCHIVO_MUT)
        archivo_mutaciones = ARCHIVO
    except FileNotFoundError:
        print(f"Error: El archivo de mutaciones '{RUTA_ARCHIVO_MUT}' no fue encontrado.")
        exit()

    # La función extract_pos_ref_mut ahora solo se usará para PARSEAR la 'pos', 'ref', 'mutation'
    # de la columna 'aa_mutation', no para generar la cadena completa.
    def extract_pos_ref_mut(aa_mutation_str):
        if not isinstance(aa_mutation_str, str):
            return None, None, None
        match = re.match(r'([A-Z])(\d+)([A-Z])', aa_mutation_str)
        if match:
            ref = match.group(1)
            pos = int(match.group(2))
            mutation = match.group(3)
            return pos, ref, mutation
        else:
            return None, None, None

    # Aplicar la función para crear las columnas 'pos', 'ref', 'mutation'
    archivo_mutaciones[['pos', 'ref', 'mutation']] = archivo_mutaciones['aa_mutation'].apply(lambda x: pd.Series(extract_pos_ref_mut(x)))

    archivo_mutaciones.dropna(subset=['pos'], inplace=True)
    archivo_mutaciones['pos'] = archivo_mutaciones['pos'].astype(int)

    resultados_clasificacion = []

    for index, mutacion_data_row in archivo_mutaciones.iterrows():
        gen = str(mutacion_data_row["Gen_names"]).upper()
        posicion = int(mutacion_data_row["pos"])
        cambio = str(mutacion_data_row["mutation"]).upper()
        
        # Obtenemos la cadena de mutación completa directamente de la columna 'aa_mutation'
        original_aa_mutation = str(mutacion_data_row["aa_mutation"])

        informacion_mutacion = clasificar_mutacion(
            gen, posicion, cambio,
            gene_domain_data_from_uniprot,
            original_aa_mutation # Pasamos la cadena original de 'aa_mutation'
        )
        resultados_clasificacion.append(informacion_mutacion)

""" 
       print(f"\n--- Clasificación para {gen} ({informacion_mutacion['mutation_RefPosMut']}) ---")
        for key, value in informacion_mutacion.items():
            if key == "dominios_afectados_detallado":
                print(f"Dominios Afectados Detallado:")
                if value:
                    for i, d in enumerate(value):
                        print(f"  - {i+1}. Motivo UniProt: {d['motif_uniprot']}, Rango: {d['range']}, Categoría Clasificador: {d['categoria_clasificador']}")
                else:
                    print("  Ninguno de los motivos de UniProt cargados solapa con esta posición.")
            else:
                print(f"{key.replace('_', ' ').capitalize()}: {value}")
        print("-" * 30)
"""

print("\n--- RESUMEN DE CLASIFICACIONES ---")
df_resultados = pd.DataFrame(resultados_clasificacion)

if 'dominios_afectados_detallado' in df_resultados.columns:
    df_resultados = df_resultados.drop(columns=['dominios_afectados_detallado'])

df_resultados['rangos_uniprot_resumen'] = df_resultados['rangos_uniprot_resumen'].fillna('N/A')
df_resultados['dominios_clasificador_resumen'] = df_resultados['dominios_clasificador_resumen'].fillna('N/A')

df_resultados['rangos_uniprot_resumen'] = df_resultados['rangos_uniprot_resumen'].apply(
    lambda x: x.split('; ') if pd.notna(x) and x != 'N/A' else [] 
)

df_resultados_exploded = df_resultados.explode('rangos_uniprot_resumen', ignore_index=True)

df_resultados_exploded.dropna(subset=['rangos_uniprot_resumen'], inplace=True)

# --- LÓGICA PARA FILTRAR 'Topological domain_Cytoplasmic' DEL DATAFRAME FINAL ---
# Esto elimina filas donde el rango de UniProt es *exactamente* 'Topological domain_Cytoplasmic: [X-Y]'
# NOTA: Asegúrate que el string 'Topological domain_Cytoplasmic: [147-509]' coincide con lo que aparece en la columna.
# Podría variar en los rangos si no es siempre el mismo.
df_final_para_mostrar = df_resultados_exploded.loc[df_resultados_exploded["rangos_uniprot_resumen"] != 'Topological domain_Cytoplasmic: [147-509]']

# --- Opcional: También puedes eliminar 'Chain' si todavía aparece en el resumen de rangos
df_final_para_mostrar = df_final_para_mostrar.loc[~df_final_para_mostrar["rangos_uniprot_resumen"].str.contains('Chain_', na=False)]


# Reordenar columnas para una vista más clara
desired_columns_order = [
    "gen",
    "mutation_RefPosMut",
    "posicion",
    "cambio_residuo",
    "dominios_clasificador_resumen",
    "rangos_uniprot_resumen",
]
existing_columns = [col for col in desired_columns_order if col in df_final_para_mostrar.columns] # Usar df_final_para_mostrar aquí
df_final_para_mostrar = df_final_para_mostrar[existing_columns]


print(f"\nTotal de mutaciones originales: {len(archivo_mutaciones)}")
print(f"Total de filas en el DataFrame expandido y filtrado: {len(df_final_para_mostrar)}") # Actualizado

print("\n--- ANÁLISIS DE MUTACIONES COMPLETADO ---")
df_final_para_mostrar.head(10)

--- INICIO DEL ANÁLISIS DE MUTACIONES ---


NameError: name 'get_uniprot_features_as_dict_for_genes' is not defined